# Movie Review Analysis Pipeline — Demo Notebook

本 Notebook 把整个工作流逐步演示一遍，对应作业的五大模块：

1. **网页抓取（2 个域 / 3 个数据集）**：YouTube 评论 + IMDb 影评 + TMDB 海报
2. **向量化对比（≥2 种方法）**：TF-IDF vs Doc2Vec（文本）+ CLIP vs ResNet50（图像）
3. **API 交互**：OpenAI GPT-4o-mini 做情感分析 + 合成评论 + Agent
4. **机器学习**：K-Means 聚类 + 情感分类 + MobileNet 海报迁移学习
5. **可视化**：matplotlib + seaborn 共 7 张图

工作流是闭环的：抓取 → 向量化 → GPT 增强 → ML → 可视化，每一步都喂给下一步。

## 0. 环境检查

In [ ]:
import sys, os, pathlib
ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project root:', ROOT)
from src.utils import load_config, env
cfg = load_config()
print('OpenAI key set?', bool(env('OPENAI_API_KEY') and not env('OPENAI_API_KEY').startswith('sk-xxxx')))
print('TMDB key set?', bool(env('TMDB_API_KEY') and not env('TMDB_API_KEY').startswith('xxxx')))
print('Genres:', list(cfg['movies'].keys()))

## 1. 数据抓取（YouTube + IMDb + TMDB）

> **注意**：完整抓取约 30-45 分钟。如果只想看流程跑通，先在终端跑 `python main.py smoke`。

In [ ]:
from src.scrapers.youtube_scraper import scrape_all_movies
df_yt = scrape_all_movies(cfg)
df_yt.head()

In [ ]:
from src.scrapers.imdb_scraper import scrape_all_imdb
df_im = scrape_all_imdb(cfg)
df_im.head()

In [ ]:
from src.scrapers.tmdb_scraper import scrape_all_tmdb
df_meta, df_img = scrape_all_tmdb(cfg)
print('Movies:', len(df_meta), 'Images:', len(df_img))
df_meta.head()

In [ ]:
from src.scrapers.merge_data import merge_reviews
df_all = merge_reviews()
df_all.head()

## 2. 向量化对比

### 2.1 文本：TF-IDF vs Word2Vec vs Doc2Vec

对比指标：**聚类纯度**（KMeans 聚类与真实 genre 的吻合度）+ **KNN mAP@10**（相似检索是否同类相邻）。

In [ ]:
from src.vectorize.text_vectorize import compare_vectorizers
vecs, metrics = compare_vectorizers(df_all, label_col='genre', text_col='text')
metrics

### 2.2 图像：CLIP vs ResNet50

在 TMDB 海报上跑相同的两个指标。

In [ ]:
from src.vectorize.image_vectorize import compare_image_vectorizers
img_mats, img_metrics = compare_image_vectorizers()
img_metrics

## 3. OpenAI GPT API 交互

### 3.1 情感分析（批量打标，从 300 条样本起步）

In [ ]:
from src.api.sentiment_analysis import annotate_dataframe
df_sent = annotate_dataframe(df_all, sample_size=cfg['openai']['sentiment_sample_size'])
df_sent[['movie_title','genre','sentiment','intensity','topic']].head(10)

### 3.2 合成评论增强

In [ ]:
from src.api.synthetic_reviews import augment_dataset
df_aug = augment_dataset(df_all, df_meta, per_genre=cfg['openai']['synthetic_per_genre'])
print('Real:', (~df_aug['is_synthetic']).sum(), '| Synthetic:', df_aug['is_synthetic'].sum())
df_aug[df_aug['is_synthetic']].head()

### 3.3 Movie Agent（ReAct 风格）

In [ ]:
from src.api.movie_agent import run_agent
print(run_agent('Compare audience sentiment of action movies vs horror movies, then recommend a movie similar to John Wick.'))

## 4. 机器学习

In [ ]:
from src.ml.text_clustering import run_full_clustering
df_clu = run_full_clustering(df_all)
df_clu[['movie_title','genre','cluster_tfidf','cluster_doc2vec']].head(10)

In [ ]:
from src.ml.sentiment_classifier import benchmark_sentiment
bench = benchmark_sentiment(df_sent)
bench

In [ ]:
from src.ml.poster_classifier import train_poster_classifier
history = train_poster_classifier()
history['history']

## 5. 可视化（7 张图一键生成）

In [ ]:
from src.viz.visualize import plot_all
plot_all()
print('图已保存到 outputs/figures/')

In [ ]:
from IPython.display import Image, display
import os
fig_dir = 'outputs/figures'
for f in sorted(os.listdir(fig_dir)):
    if f.endswith('.png'):
        print(f)
        display(Image(os.path.join(fig_dir, f)))